# Tahap 0 — Verifikasi RSA (Representational Similarity Analysis)

Notebook ini menjawab pertanyaan gerbang dari
`notes/research_question/03_pivot2_group_consistency.md` §12.4 /
`notes/simple_notes.md` §3-4:

> Kalau dua demografi (atau dua pertanyaan) beda tapi jawaban survei aslinya
> mirip — apakah representasi/embedding mereka di "otak" LLM juga berdekatan?

- Kalau **iya** (RSA lolos) → boleh lanjut desain `L_group`/`L_question` pakai
  bobot dari representasi LLM langsung (Tahap 1).
- Kalau **tidak** (RSA gagal) → representasi mentah LLM tidak bisa dipercaya
  buat ini, harus fallback ke bobot yang dipelajari secara statistik.

**Model yang dipakai:** `mistralai/Mistral-7B-v0.1` — dipilih karena ini
satu-satunya model yang dipakai **persis sama** di dua paper acuan kita
(SubPOP dan llm-opinions / "What Do Large Language Models Know About
Opinions?"), jadi kita bisa langsung pakai temuan layer-choice mereka tanpa
re-derive dari nol.

**Cara pakai:** cek dulu setting Kaggle di cell markdown berikutnya, lalu
`Run All`. Tidak ada yang perlu diedit manual kecuali path data ketemunya
beda dari dugaan otomatis di bawah.

## Sebelum jalan: setting Kaggle yang wajib dicek

1. **Accelerator**: menu titik-tiga kanan atas → *Notebook options* → pilih
   **GPU T4 x2** atau **GPU P100** (bukan CPU / TPU).
2. **Internet**: di panel yang sama, nyalakan **Internet: On** (dibutuhkan
   buat download model Mistral-7B-v0.1 dari HuggingFace, ±14GB, sekali di awal).
3. **Upload data** — notebook ini butuh file `opinionqa.csv` dari repo
   (path asli: `datasets/subpop/data/opinionqa/processed/opinionqa.csv`):
   - Panel kanan → **Add Data** → **Upload** → pilih file `opinionqa.csv` →
     beri nama dataset → **Create**.
   - Setelah ke-*attach*, otomatis muncul di
     `/kaggle/input/<nama-dataset>/opinionqa.csv` dan cell config di bawah
     bakal otomatis nemuin-nya sendiri (cari pola `**/opinionqa.csv`).
   - Kalau ternyata nggak ketemu otomatis, isi manual variabel
     `OPINIONQA_CSV_PATH` di cell config.

Perkiraan total waktu jalan: **15–25 menit** (sebagian besar buat download +
load model sekali di awal; ekstraksi representasi buat ratusan prompt cuma
beberapa menit, sisanya hitungan statistik yang ringan).

In [ ]:
# Idempotent — aman dijalankan ulang.
!pip install -q -U "transformers>=4.44" accelerate scipy scikit-learn tqdm
!pip install -q -U bitsandbytes  # cuma kepake kalau USE_4BIT=True di cell config

In [ ]:
import os
import ast
import glob
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, spearmanr
from sklearn.metrics import pairwise_distances
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    raise RuntimeError(
        "GPU tidak terdeteksi. Cek menu titik-tiga kanan atas -> Notebook options -> "
        "Accelerator -> pilih GPU T4 x2 atau P100, lalu restart session & Run All lagi."
    )

In [ ]:
# ── Model ────────────────────────────────────────────────────────
MODEL_PATH = "mistralai/Mistral-7B-v0.1"
USE_4BIT = False   # set True kalau kena OOM (misal cuma dapat jatah 1x T4 16GB)

# ── Data ─────────────────────────────────────────────────────────
# Cari otomatis file opinionqa.csv di antara Kaggle Dataset yang di-attach.
_candidates = glob.glob("/kaggle/input/**/opinionqa.csv", recursive=True)
if _candidates:
    OPINIONQA_CSV_PATH = _candidates[0]
elif os.path.exists("opinionqa.csv"):
    OPINIONQA_CSV_PATH = "opinionqa.csv"
else:
    raise FileNotFoundError(
        "Tidak ketemu opinionqa.csv. Upload file ini sebagai Kaggle Dataset dulu "
        "(Add Data -> Upload -> opinionqa.csv), attach ke notebook, lalu jalankan ulang "
        "cell ini. Kalau nama foldernya lain dari dugaan, isi manual: "
        "OPINIONQA_CSV_PATH = '/kaggle/input/.../opinionqa.csv'"
    )
print("Pakai data dari:", OPINIONQA_CSV_PATH)

# ── Cakupan analisis ─────────────────────────────────────────────
# None = pakai SEMUA data yang ada (lebih akurat, masih cepat kok).
# Isi angka (misal 100) kalau cuma mau coba cepat dulu / debug.
N_QKEYS_FOR_GROUP_WD = None   # jumlah pertanyaan dipakai hitung jarak-asli antar KELOMPOK
N_QUESTIONS_SAMPLE   = None   # jumlah pertanyaan yang jadi anggota sumbu PERTANYAAN itu sendiri

RANDOM_SEED = 42
N_PERMUTATIONS = 2000  # buat uji signifikansi (permutation / Mantel test) di layer terbaik

# ── Output ───────────────────────────────────────────────────────
# WAJIB di /kaggle/working -- itu satu-satunya folder yang boleh ditulis.
# /kaggle/input (tempat dataset yang di-upload) selalu read-only.
OUT_DIR = "/kaggle/working/tahap0_rsa"
assert not OUT_DIR.startswith("/kaggle/input"), (
    "OUT_DIR nggak boleh di /kaggle/input -- itu read-only (tempat dataset yang di-upload). "
    "Tempat nulis file yang boleh cuma di bawah /kaggle/working/."
)
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

In [ ]:
df = pd.read_csv(OPINIONQA_CSV_PATH)

def _parse_list(x):
    return ast.literal_eval(x) if isinstance(x, str) else x

df["responses"] = df["responses"].apply(_parse_list)
df["ordinal"]   = df["ordinal"].apply(_parse_list)
df["options"]   = df["options"].apply(_parse_list)

# Buang baris "Overall" — itu agregat semua orang, bukan kelompok demografi.
df = df[df["attribute"] != "Overall"].reset_index(drop=True)
df["group_label"] = df["attribute"] + ": " + df["group"].astype(str)

print(f"Total baris: {len(df)}")
print(f"Jumlah kelompok demografi unik: {df['group_label'].nunique()}")
print(f"Jumlah pertanyaan (qkey) unik: {df['qkey'].nunique()}")

In [ ]:
def ordinal_weighted_mean(responses, ordinal):
    """Skor rata-rata satu kelompok di skala pertanyaan itu
    (skor besar = condong ke opsi yang kodenya besar)."""
    r = np.asarray(responses, dtype=float)
    o = np.asarray(ordinal, dtype=float)
    if r.sum() == 0:
        return np.nan
    return float(np.sum(r * o) / r.sum())

df["scalar_score"] = df.apply(lambda row: ordinal_weighted_mean(row["responses"], row["ordinal"]), axis=1)

GROUP_LABELS = sorted(df["group_label"].unique().tolist())
print(f"{len(GROUP_LABELS)} kelompok demografi:")
print(GROUP_LABELS)

# tabel (kelompok x pertanyaan) berisi skor skalar itu -- dipakai di sumbu PERTANYAAN nanti
scalar_pivot = df.pivot_table(index="group_label", columns="qkey", values="scalar_score")
print("\nBentuk tabel skor (kelompok x pertanyaan):", scalar_pivot.shape)

## 1. Hitung "jarak-asli" (dari data survei manusia)

- **Sumbu kelompok**: rata-rata Wasserstein Distance (metrik jarak yang
  sama persis dipakai SubPOP) antara distribusi jawaban dua kelompok,
  dirata-rata lewat semua pertanyaan yang sama-sama mereka jawab.
- **Sumbu pertanyaan**: dua pertanyaan dianggap "mirip secara logika" kalau
  pola jawabannya **lintas-kelompok** mirip (misal: kalau golongan yang sama
  yang cenderung setuju di pertanyaan A juga cenderung setuju di pertanyaan
  B, walau topiknya beda) — diukur lewat korelasi antar-kolom di tabel skor
  di atas, lalu `jarak = 1 - korelasi`.

In [ ]:
resp_lookup = {
    (gl, qk): (resp, ordv)
    for gl, qk, resp, ordv in zip(df["group_label"], df["qkey"], df["responses"], df["ordinal"])
}

rng = np.random.default_rng(RANDOM_SEED)
all_qkeys = df["qkey"].unique().tolist()

if N_QKEYS_FOR_GROUP_WD is None:
    sample_qkeys_for_wd = all_qkeys
else:
    sample_qkeys_for_wd = list(rng.choice(all_qkeys, size=min(N_QKEYS_FOR_GROUP_WD, len(all_qkeys)), replace=False))

n_g = len(GROUP_LABELS)
group_real_dist = np.full((n_g, n_g), np.nan)

for i in tqdm(range(n_g), desc="Hitung jarak-asli antar-kelompok"):
    for j in range(i, n_g):
        if i == j:
            group_real_dist[i, j] = 0.0
            continue
        gi, gj = GROUP_LABELS[i], GROUP_LABELS[j]
        wds = []
        for qk in sample_qkeys_for_wd:
            ri = resp_lookup.get((gi, qk))
            rj = resp_lookup.get((gj, qk))
            if ri is None or rj is None:
                continue
            respA, ordA = ri
            respB, ordB = rj
            if len(ordA) != len(ordB):
                continue  # jaga-jaga kalau skala opsinya beda (harusnya jarang terjadi)
            wds.append(wasserstein_distance(ordA, ordB, u_weights=respA, v_weights=respB))
        group_real_dist[i, j] = group_real_dist[j, i] = float(np.mean(wds)) if wds else np.nan

print("\nContoh: 6 kelompok paling mirip sama kelompok pertama menurut jawaban asli:")
print(pd.Series(group_real_dist[0], index=GROUP_LABELS).sort_values().head(7))

In [ ]:
if N_QUESTIONS_SAMPLE is None:
    QUESTION_SAMPLE = all_qkeys
else:
    QUESTION_SAMPLE = list(rng.choice(all_qkeys, size=min(N_QUESTIONS_SAMPLE, len(all_qkeys)), replace=False))

question_corr = scalar_pivot[QUESTION_SAMPLE].corr(method="pearson", min_periods=10)
question_real_dist = (1.0 - question_corr.values)

qtext_lookup = df.drop_duplicates("qkey").set_index("qkey")[["question", "options"]].to_dict("index")

print(f"{len(QUESTION_SAMPLE)} pertanyaan dipakai buat sumbu pertanyaan.")
print("Bentuk matriks jarak-asli antar-pertanyaan:", question_real_dist.shape)

## 2. Bangun prompt buat tiap kelompok & tiap pertanyaan

Prompt kelompok cuma deskripsi demografi polos (tanpa pertanyaan survei apa
pun) — supaya representasi yang kita ambil murni "konsep kelompok itu",
bukan tercampur konteks satu pertanyaan tertentu. Prompt pertanyaan pakai
format standar survei (pertanyaan + opsi berlabel huruf + `Answer:`),
konsisten dengan cara SubPOP/llm-opinions mem-format pertanyaan survei ke LLM.

In [ ]:
def build_group_prompt(group_label):
    attribute, group = group_label.split(": ", 1)
    return f"Survey respondent profile.\n{attribute}: {group}."

def build_question_prompt(qkey):
    info = qtext_lookup[qkey]
    q, opts = info["question"], info["options"]
    letters = [chr(ord("A") + i) for i in range(len(opts))]
    lines = [q] + [f"{letter}. {opt}" for letter, opt in zip(letters, opts)] + ["Answer:"]
    return "\n".join(lines)

group_prompts = {g: build_group_prompt(g) for g in GROUP_LABELS}
question_prompts = {q: build_question_prompt(q) for q in QUESTION_SAMPLE}

print("Contoh prompt KELOMPOK:\n" + list(group_prompts.values())[0])
print("\n---\n")
print("Contoh prompt PERTANYAAN:\n" + list(question_prompts.values())[0])

## 3. Load model & ambil representasi internal

Kita ambil representasi token **terakhir** di **setiap layer** (`output_hidden_states=True`
dari Transformers biasa) — ini bahan mentah yang sama seperti yang dipakai
`llm_opinions/utils/activation_utils.py` (probing) di paper llm-opinions,
cuma ditulis ulang lebih ringkas di sini karena kita tidak butuh fitur
per-attention-head atau steering hook mereka, cukup representasi per-layer.

In [ ]:
print(f"Loading tokenizer & model: {MODEL_PATH} (download pertama kali bisa beberapa menit, ~14GB)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = dict(torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True)
if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs.pop("torch_dtype", None)
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16
    )

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **model_kwargs)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
print(f"Model loaded. Jumlah layer: {NUM_LAYERS} (+1 untuk embedding awal sebelum layer ke-1)")

In [ ]:
@torch.no_grad()
def get_hidden_states_all_layers(prompt: str) -> np.ndarray:
    """Representasi token TERAKHIR di setiap layer.
    Output shape: (num_layers+1, hidden_size) -- index 0 = embedding awal.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs, output_hidden_states=True)
    hs = torch.stack(out.hidden_states, dim=0)   # (num_layers+1, batch=1, seq_len, hidden)
    return hs[:, 0, -1, :].float().cpu().numpy()  # (num_layers+1, hidden)

group_embeddings = {}
for g, prompt in tqdm(group_prompts.items(), desc="Ekstraksi representasi kelompok"):
    group_embeddings[g] = get_hidden_states_all_layers(prompt)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

question_embeddings = {}
for q, prompt in tqdm(question_prompts.items(), desc="Ekstraksi representasi pertanyaan"):
    question_embeddings[q] = get_hidden_states_all_layers(prompt)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

group_emb_array = np.stack([group_embeddings[g] for g in GROUP_LABELS])
question_emb_array = np.stack([question_embeddings[q] for q in QUESTION_SAMPLE])

np.savez(
    os.path.join(OUT_DIR, "embeddings_raw.npz"),
    group_emb=group_emb_array,
    question_emb=question_emb_array,
    group_labels=np.array(GROUP_LABELS, dtype=object),
    question_ids=np.array(QUESTION_SAMPLE, dtype=object),
)
print("Representasi mentah disimpan ke", os.path.join(OUT_DIR, "embeddings_raw.npz"))

## 4. RSA: bandingkan jarak-representasi-LLM vs jarak-asli

Statistik intinya: korelasi **Spearman** antara dua matriks jarak (ini
definisi RSA dari Kriegeskorte et al. 2008 — lihat
`notes/research_question/03_pivot2_group_consistency.md` §12.5). Buat
p-value, dipakai **permutation test** (acak urutan matriks berkali-kali,
lihat seberapa sering hasil se-random itu >= hasil asli kita), bukan
p-value Spearman standar — soalnya entri di dalam satu matriks jarak itu
saling berkaitan (bukan sampel independen), jadi p-value standar bakal
kelewat percaya diri. Ini teknik standar di RSA, namanya **Mantel test**.

In [ ]:
def upper_tri(mat):
    idx = np.triu_indices_from(mat, k=1)
    return mat[idx]

def rho_only(emb_array, real_dist_matrix, layer_idx):
    rep_dist = pairwise_distances(emb_array[:, layer_idx, :], metric="cosine")
    real_flat, rep_flat = upper_tri(real_dist_matrix), upper_tri(rep_dist)
    mask = ~np.isnan(real_flat) & ~np.isnan(rep_flat)
    rho, _ = spearmanr(real_flat[mask], rep_flat[mask])
    return rho

def rsa_with_permutation_test(emb_array, real_dist_matrix, layer_idx, n_perm=N_PERMUTATIONS, seed=RANDOM_SEED):
    rep_dist = pairwise_distances(emb_array[:, layer_idx, :], metric="cosine")
    real_flat, rep_flat = upper_tri(real_dist_matrix), upper_tri(rep_dist)
    mask = ~np.isnan(real_flat) & ~np.isnan(rep_flat)
    real_flat, rep_flat = real_flat[mask], rep_flat[mask]
    rho, _ = spearmanr(real_flat, rep_flat)

    rng_p = np.random.default_rng(seed)
    n = real_dist_matrix.shape[0]
    perm_rhos = np.empty(n_perm)
    for k in range(n_perm):
        perm = rng_p.permutation(n)
        permuted_flat = upper_tri(real_dist_matrix[np.ix_(perm, perm)])[mask]
        r, _ = spearmanr(permuted_flat, rep_flat)
        perm_rhos[k] = 0.0 if np.isnan(r) else r

    p_value = float(np.mean(np.abs(perm_rhos) >= abs(rho)))
    return float(rho), p_value

n_layers_total = group_emb_array.shape[1]
group_rhos    = [rho_only(group_emb_array, group_real_dist, L) for L in range(n_layers_total)]
question_rhos = [rho_only(question_emb_array, question_real_dist, L) for L in range(n_layers_total)]

results_df = pd.DataFrame({
    "layer": list(range(n_layers_total)),
    "rsa_rho_group_axis": group_rhos,
    "rsa_rho_question_axis": question_rhos,
})
print(results_df.to_string())

In [ ]:
best_group_layer    = int(np.nanargmax(np.abs(group_rhos)))
best_question_layer = int(np.nanargmax(np.abs(question_rhos)))

print(f"Layer terbaik sumbu KELOMPOK  : {best_group_layer}  (rho={group_rhos[best_group_layer]:.3f})")
print(f"Layer terbaik sumbu PERTANYAAN: {best_question_layer}  (rho={question_rhos[best_question_layer]:.3f})")
print(f"\nMenjalankan permutation test ({N_PERMUTATIONS}x acak) di layer terbaik masing-masing sumbu...")

group_rho_best, group_p_best = rsa_with_permutation_test(group_emb_array, group_real_dist, best_group_layer)
question_rho_best, question_p_best = rsa_with_permutation_test(question_emb_array, question_real_dist, best_question_layer)

print(f"\n[SUMBU KELOMPOK]   layer {best_group_layer}: rho={group_rho_best:.3f}, p={group_p_best:.4f}")
print(f"[SUMBU PERTANYAAN] layer {best_question_layer}: rho={question_rho_best:.3f}, p={question_p_best:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(results_df["layer"], results_df["rsa_rho_group_axis"], marker="o", label="Sumbu kelompok (demografi)")
ax.plot(results_df["layer"], results_df["rsa_rho_question_axis"], marker="s", label="Sumbu pertanyaan")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(15, color="red", linestyle="--", alpha=0.5, label="layer ~15 (batas 'cukup dalam' menurut llm-opinions)")
ax.set_xlabel("Layer")
ax.set_ylabel("RSA Spearman rho\n(jarak-representasi-LLM vs jarak-jawaban-asli)")
ax.set_title("Tahap 0 -- RSA check: apakah representasi LLM cocok sama kemiripan jawaban survei asli?")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "rsa_rho_vs_layer.png"), dpi=150)
plt.show()

In [ ]:
def verdict(rho, p, label, loss_name):
    if p >= 0.05:
        return (f"[GAGAL - tidak signifikan] {label}: rho={rho:.3f}, p={p:.4f} -- tidak ada bukti hubungan "
                f"yang bisa diandalkan (bisa jadi kebetulan acak).")
    if rho > 0.15:
        return (f"[LOLOS] {label}: rho={rho:.3f}, p={p:.4f} -- representasi LLM cukup dipercaya "
                f"jadi bobot kedekatan untuk {loss_name}.")
    if rho < -0.15:
        return (f"[GAGAL - ARAH TERBALIK] {label}: rho={rho:.3f}, p={p:.4f} -- ada hubungan yang kuat dan "
                f"signifikan, TAPI ARAHNYA TERBALIK dari yang diharapkan (representasi makin beda justru "
                f"buat kelompok/pertanyaan yang jawaban aslinya makin mirip). JANGAN langsung dipakai jadi "
                f"bobot kedekatan sebelum diselidiki -- bisa jadi artefak desain prompt, bisa juga temuan asli.")
    return (f"[SIGNIFIKAN TAPI LEMAH] {label}: rho={rho:.3f}, p={p:.4f} -- ada sinyal tipis, kemungkinan "
            f"cuma kedeteksi gara-gara sampelnya besar (bukan hubungan yang kuat secara praktis). "
            f"Pertimbangkan gabung dengan bobot statistik, jangan andalkan representasi mentah sepenuhnya.")

print(verdict(group_rho_best, group_p_best, "Sumbu kelompok", "L_group"))
print(verdict(question_rho_best, question_p_best, "Sumbu pertanyaan", "L_question"))

In [ ]:
results_df.to_csv(os.path.join(OUT_DIR, "rsa_rho_per_layer.csv"), index=False)

summary = pd.DataFrame([
    {"axis": "group", "best_layer": best_group_layer, "rho": group_rho_best, "p_value": group_p_best,
     "n_items": len(GROUP_LABELS)},
    {"axis": "question", "best_layer": best_question_layer, "rho": question_rho_best, "p_value": question_p_best,
     "n_items": len(QUESTION_SAMPLE)},
])
summary.to_csv(os.path.join(OUT_DIR, "rsa_summary.csv"), index=False)
print(summary.to_string(index=False))

print(f"\nSemua output ada di: {OUT_DIR}")
print("Download folder ini dari tab 'Output' Kaggle sebelum session berakhir,")
print("lalu bawa angkanya balik untuk update research_question/03_pivot2_group_consistency.md paragraf 12.9 / TODO.md.")

## Selesai — cara baca hasilnya

- **rho (Spearman correlation)**: makin besar & positif, makin cocok jarak
  representasi LLM sama jarak jawaban asli. Rentang -1 sampai 1.
- **p-value**: dari *permutation test* (Mantel test), bukan p-value Spearman
  standar — lihat penjelasan di atas cell RSA.
- Ambang di cell verdict (**p < 0.05 dan rho > 0.15 → LOLOS**) itu titik
  awal yang wajar, bukan angka sakral. Kalau hasilnya di sekitar ambang itu
  (misal rho=0.14, p=0.04), baca angkanya manual, jangan cuma percaya label
  otomatis.

**Langkah selanjutnya:**
1. Download folder `/kaggle/working/tahap0_rsa/` (tombol *Output* di panel kanan).
2. Bawa `rsa_summary.csv` + `rsa_rho_vs_layer.png` balik ke repo lokal.
3. Update `notes/research_question/03_pivot2_group_consistency.md` §12.9
   (checklist Tahap 0) dan `TODO.md` PRIORITAS #1 dengan hasilnya —
   lolos/gagal, di layer berapa, untuk kedua sumbu.
4. Kalau **lolos** → lanjut ke Tahap 1 (desain `L_group`/`L_question`).
   Kalau **gagal** → baca `notes/simple_notes.md` bagian 4, "skenario RSA
   salah", untuk rencana fallback-nya.

## Tahap 0b (diagnostik) — apakah rho negatif sumbu kelompok itu artefak prompt?

Hasil run pertama: sumbu kelompok dapat **rho negatif yang signifikan**
(bukan cuma lemah/nol). Sebelum menyimpulkan "representasi LLM salah paham
soal kemiripan demografi", kita cek dulu kemungkinan yang lebih murah:
jangan-jangan itu cuma artefak template prompt `"{ATTRIBUTE}: {group}."`
— karena kelompok-kelompok dalam **atribut yang sama** (misal 5 kelompok
`POLIDEOLOGY`) teksnya nyaris identik secara permukaan (cuma beda 1-2 kata),
jadi representasinya otomatis deket, **padahal** kelompok dalam satu atribut
sering justru paling beda pendapatnya (misal Very Liberal vs Very
Conservative).

Cell di bawah bikin ulang prompt kelompok pakai kalimat natural (nggak
nyebut kode atribut internal survei kayak `AGE:`/`POLIDEOLOGY:`), lalu
ekstrak ulang representasinya **pakai model yang sama yang sudah ke-load**
(nggak perlu load ulang), dan hitung ulang RSA-nya cuma buat sumbu kelompok.
Jarak-asli (`group_real_dist`) tidak berubah — cuma sisi representasinya
yang diganti.

In [ ]:
GROUP_TEMPLATES_V2 = {
    "AGE": lambda g: f"This survey respondent is {g} years old.",
    "CITIZEN": lambda g: f"This survey respondent is {'a U.S. citizen' if g == 'Yes' else 'not a U.S. citizen'}.",
    "CREGION": lambda g: f"This survey respondent lives in the {g} region of the United States.",
    "EDUCATION": lambda g: f"This survey respondent's highest level of education is {g}.",
    "INCOME": lambda g: f"This survey respondent's household income is {g}.",
    "MARITAL": lambda g: f"This survey respondent's marital status is {g}.",
    "POLIDEOLOGY": lambda g: f"Politically, this survey respondent describes their views as {g.lower()}.",
    "POLPARTY": lambda g: f"This survey respondent's political party affiliation is {g}.",
    "RACE": lambda g: f"This survey respondent's race is {g}.",
    "RELIG": lambda g: f"This survey respondent's religion is {g}.",
    "RELIGATTEND": lambda g: f"This survey respondent attends religious services: {g.lower()}.",
    "SEX": lambda g: f"This survey respondent is {g.lower()}.",
}

def build_group_prompt_v2(group_label):
    attribute, group = group_label.split(": ", 1)
    return GROUP_TEMPLATES_V2[attribute](group)

group_prompts_v2 = {g: build_group_prompt_v2(g) for g in GROUP_LABELS}

print("Contoh perbandingan prompt v1 (lama) vs v2 (natural, tanpa nama-atribut):\n")
for g in GROUP_LABELS[:4] + ["POLIDEOLOGY: Very liberal", "POLIDEOLOGY: Very conservative"]:
    print(f"v1: {group_prompts[g]!r}")
    print(f"v2: {group_prompts_v2[g]!r}")
    print()

In [ ]:
group_embeddings_v2 = {}
for g, prompt in tqdm(group_prompts_v2.items(), desc="Ekstraksi representasi kelompok (v2, natural)"):
    group_embeddings_v2[g] = get_hidden_states_all_layers(prompt)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

group_emb_array_v2 = np.stack([group_embeddings_v2[g] for g in GROUP_LABELS])

np.savez(
    os.path.join(OUT_DIR, "embeddings_group_v2.npz"),
    group_emb_v2=group_emb_array_v2,
    group_labels=np.array(GROUP_LABELS, dtype=object),
)
print("Representasi v2 disimpan ke", os.path.join(OUT_DIR, "embeddings_group_v2.npz"))

In [ ]:
group_rhos_v2 = [rho_only(group_emb_array_v2, group_real_dist, L) for L in range(n_layers_total)]

compare_df = pd.DataFrame({
    "layer": list(range(n_layers_total)),
    "rho_v1_attr_code_prefix": group_rhos,
    "rho_v2_natural_sentence": group_rhos_v2,
})
print(compare_df.to_string())

best_group_layer_v2 = int(np.nanargmax(np.abs(group_rhos_v2)))
group_rho_best_v2, group_p_best_v2 = rsa_with_permutation_test(group_emb_array_v2, group_real_dist, best_group_layer_v2)

print(f"\n[v1 - format 'ATTR: value']  layer {best_group_layer}: rho={group_rho_best:.3f}, p={group_p_best:.4f}")
print(f"[v2 - kalimat natural]       layer {best_group_layer_v2}: rho={group_rho_best_v2:.3f}, p={group_p_best_v2:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(compare_df["layer"], compare_df["rho_v1_attr_code_prefix"], marker="o", label="v1 -- format 'ATTR: value'")
ax.plot(compare_df["layer"], compare_df["rho_v2_natural_sentence"], marker="o", label="v2 -- kalimat natural")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("Layer")
ax.set_ylabel("RSA Spearman rho (sumbu kelompok)")
ax.set_title("Tahap 0b -- pengaruh desain prompt terhadap rho sumbu kelompok")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "rsa_group_prompt_comparison.png"), dpi=150)
plt.show()

compare_df.to_csv(os.path.join(OUT_DIR, "rsa_group_prompt_comparison.csv"), index=False)

print(verdict(group_rho_best_v2, group_p_best_v2, "Sumbu kelompok (prompt v2, natural)", "L_group"))
print()
if group_rho_best < -0.1 and group_rho_best_v2 > 0.05:
    print("=> Tanda berbalik dari negatif (v1) ke positif (v2) -- kuat dugaan v1 memang artefak template prompt, "
          "bukan LLM salah paham soal kemiripan demografi. Lanjut pakai prompt gaya v2 buat Tahap 1.")
elif group_rho_best_v2 < -0.1:
    print("=> Masih negatif & cukup kuat di v2 juga -- dugaan artefak prompt TIDAK terbukti, kemungkinan ini "
          "temuan asli: representasi LLM memang belum menangkap kemiripan demografi dengan benar untuk model ini.")
else:
    print("=> Hasilnya berubah tapi tidak sepenuhnya jelas (nggak jelas positif kuat atau negatif kuat) -- "
          "baca angkanya manual, kemungkinan gabungan artefak prompt DAN keterbatasan representasi.")